# Generate LIARArg silver labels via gpt-oss-120b (Cerebras)Runs 2,123 LIARArg training articles through gpt-oss-120b via the Cerebras API, capturing both the extractive parse and the model's Chain of Thought reasoning. Writes to `phase2_data/silver/liararg_train_silver.jsonl`.Requires a `CEREBRAS_API_KEY` (or multiple keys for rotation) in the environment. Free tier at https://cloud.cerebras.ai.

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate

export CEREBRAS_KEYS="<CEREBRAS_API_KEY>,
    <CEREBRAS_API_KEY>,
    <CEREBRAS_API_KEY>,
    <CEREBRAS_API_KEY>,
    <CEREBRAS_API_KEY>,
    <CEREBRAS_API_KEY>,
    <CEREBRAS_API_KEY>,
    <CEREBRAS_API_KEY>"

mkdir -p phase2_data/silver eval_logs

nohup python3 -u <<'PY' > eval_logs/v4_liararg_silver.log 2>&1 &
import sys, json, time, os
sys.path.insert(0, '.')
from pathlib import Path
from src.phase2.config import TeacherConfig
from src.phase2.teacher import build_teacher
from src.phase2.dataset import read_jsonl

KEYS = [k.strip() for k in os.environ.get("CEREBRAS_KEYS", "").split(",") if k.strip()]
if not KEYS: raise RuntimeError("Set CEREBRAS_KEYS")
print(f"[v4-silver] {len(KEYS)} keys loaded", flush=True)

key_idx = [0]
key_dead = [False] * len(KEYS)
teacher_state = [None]
key_stats = [{"calls": 0, "errors": 0, "tpd_hit": False} for _ in KEYS]

CFG = TeacherConfig(
    backend="cerebras", model="gpt-oss-120b",
    max_input_chars=2000, max_output_tokens=2048,
    rpm_cap=30, max_requests_per_day=1000, retries_per_call=2,
)

def build_with_current_key():
    os.environ["CEREBRAS_API_KEY"] = KEYS[key_idx[0]]
    teacher_state[0] = build_teacher(CFG)

def rotate_key(reason=""):
    print(f"  [rotate] from key #{key_idx[0]+1} — {reason}", flush=True)
    for _ in range(len(KEYS)):
        key_idx[0] = (key_idx[0] + 1) % len(KEYS)
        if not key_dead[key_idx[0]]:
            print(f"  [rotate] → key #{key_idx[0]+1}", flush=True)
            build_with_current_key()
            return True
    print(f"  [rotate] all keys dead — sleeping 5 min", flush=True)
    time.sleep(300)
    for i in range(len(KEYS)):
        key_dead[i] = False
        key_stats[i]["tpd_hit"] = False
    key_idx[0] = 0
    build_with_current_key()
    return True

def annotate_with_rotation(text, source, max_attempts=None):
    if max_attempts is None: max_attempts = len(KEYS) * 3
    for attempt in range(max_attempts):
        try:
            pred, reasoning = teacher_state[0].annotate(text, source=source)
            key_stats[key_idx[0]]["calls"] += 1
            if "[teacher: daily cap reached]" in reasoning:
                key_dead[key_idx[0]] = True
                key_stats[key_idx[0]]["tpd_hit"] = True
                rotate_key("daily cap")
                continue
            if "[teacher rate-limit:" in reasoning:
                rotate_key("rate-limit signal")
                continue
            if "[teacher error:" in reasoning:
                err_low = reasoning.lower()
                if "401" in err_low or "wrong" in err_low or "auth" in err_low:
                    key_dead[key_idx[0]] = True
                    rotate_key("auth error")
                    continue
                key_stats[key_idx[0]]["errors"] += 1
                return pred, reasoning
            return pred, reasoning
        except RuntimeError as e:
            msg = str(e).lower()
            if "credits" in msg or "402" in msg or "payment" in msg:
                key_dead[key_idx[0]] = True
                rotate_key(f"hard error")
                continue
            raise
        except Exception as e:
            key_stats[key_idx[0]]["errors"] += 1
            err_str = str(e).lower()
            if "401" in err_str or "auth" in err_str or "wrong" in err_str:
                key_dead[key_idx[0]] = True
                rotate_key("auth")
                continue
            print(f"  [error] key #{key_idx[0]+1}: {e}", flush=True)
            time.sleep(2)
    return ({"claim_components":[],"premise_components":[],"citation_components":[],"relations":[]},
            "[v4-silver: max attempts exhausted]")

build_with_current_key()
print(f"[v4-silver] starting with key #{key_idx[0]+1}", flush=True)

TRAIN = Path("phase2_data/unified/liararg_train.jsonl")
OUT = Path("phase2_data/silver/liararg_train_silver.jsonl")
OUT.parent.mkdir(parents=True, exist_ok=True)
recs = read_jsonl(TRAIN)
print(f"[v4-silver] {len(recs)} LIARArg train rows", flush=True)

done_ids = set()
if OUT.exists():
    with open(OUT) as f:
        for line in f:
            try: done_ids.add(int(json.loads(line)["liararg_row_id"]))
            except: pass
print(f"[v4-silver] resuming — {len(done_ids)} silver rows on disk", flush=True)

todo = [r for r in recs
        if int(r.get("liararg_row_id", -1)) not in done_ids
        and int(r.get("liararg_row_id", -1)) >= 0]
print(f"[v4-silver] {len(todo)} rows to process", flush=True)

t0 = time.time(); n_done = 0; n_empty = 0; n_with_reasoning = 0
with open(OUT, "a") as fout:
    for rec in todo:
        rid = int(rec["liararg_row_id"])
        text = rec["input"]
        pred, reasoning = annotate_with_rotation(text, source="liararg_silver")
        is_empty = (len(pred.get("claim_components", []))==0
                    and len(pred.get("premise_components", []))==0)
        has_reasoning = bool(reasoning and not reasoning.startswith("[teacher"))
        if is_empty: n_empty += 1
        if has_reasoning: n_with_reasoning += 1
        fout.write(json.dumps({
            "instruction": rec["instruction"],
            "input": text,
            "reasoning": reasoning,
            "output": pred,
            "source_dataset": "liararg_silver",
            "label_kind": "silver",
            "split": "train",
            "domain": "politics_liararg",
            "liararg_row_id": rid,
            "teacher_model": "gpt-oss-120b",
        }, ensure_ascii=False) + "\n")
        fout.flush(); os.fsync(fout.fileno())
        n_done += 1
        if n_done % 25 == 0:
            elapsed = time.time() - t0
            rate = n_done / max(elapsed, 1e-6)
            eta = (len(todo) - n_done) / max(rate, 1e-6) / 3600
            alive = sum(1 for d in key_dead if not d)
            print(f"  {n_done}/{len(todo)}  ({elapsed/60:.0f}min, {rate:.2f}/s, "
                  f"ETA {eta:.1f}h)  alive_keys={alive}/{len(KEYS)}  "
                  f"empty={n_empty}/{n_done} ({100*n_empty/n_done:.0f}%)  "
                  f"with_reasoning={n_with_reasoning}/{n_done}", flush=True)

print("\n=== v4 silver gen DONE ===", flush=True)
print(f"  total: {n_done}  empty: {n_empty} ({100*n_empty/max(n_done,1):.1f}%)  "
      f"with_reasoning: {n_with_reasoning} ({100*n_with_reasoning/max(n_done,1):.1f}%)", flush=True)
for i, s in enumerate(key_stats, 1):
    print(f"  key #{i}: calls={s['calls']} errors={s['errors']} tpd_hit={s['tpd_hit']}", flush=True)
PY
echo "PID: $!"
sleep 10
tail -15 eval_logs/v4_liararg_silver.log